In [3]:
# Importing Libraries #
import pandas as pd
import numpy as np

# QUESTION 1 #
# 1A: Reading the stocks file into data frame sp_stks and printing column names and shape #
sp_stks = pd.read_csv('Stocks_spread.csv', encoding='latin-1')
sp_stks.columns

Index(['symbol', 'sector', 'industry', 'price', 'avg_volume', 'avg_sp',
       'market_cap'],
      dtype='object')

In [4]:
sp_stks.shape

(5611, 7)

In [5]:
# 1B: Dropping missing values from sp_stks and creating the rel_sp column #
sp_stks = sp_stks.dropna()
sp_stks['rel_sp'] = sp_stks['avg_sp'] / sp_stks['price']
sp_stks.shape

(5540, 8)

In [6]:
# 1C: Using the describe method to examine the statistical distribution of price, avg_sp, and rel_sp #
sp_stks[['price', 'avg_sp', 'rel_sp']].describe()
## Question: Does it make sense to have rel_sp > 1? ##
    ## No, it does not make sense because rel_sp represents the spread as a fraction of price. ##
## If rel_sp > 1, the spread would be larger than the stock price itself, which is unrealistic in the real world. ##

,price,avg_sp,rel_sp
count,5540.000000,5540.000000,5540.000000
mean,47.609490,0.255003,0.009655
std,131.910772,1.676051,0.104221
min,0.478800,0.002170,0.000101
25%,8.887500,0.029292,0.002057
50%,17.959950,0.081854,0.004337
75%,46.220001,0.218497,0.009289
max,4624.620117,107.982356,7.631262


In [7]:
# 1d. Creating an outliers dataframe with rel_sp > 0.1 #
outliers = sp_stks[sp_stks['rel_sp'] > 0.1]
outliers.shape[0]
## Question: How many stocks are in this data frame? ##
    ## 19 ##

19

In [8]:
outliers
## Comment on outliers:
    ## These 19 stocks have very high relative spreads (>10% of price). ##
## Most are low-priced stocks (under $30) with poor liquidity. ##
## Many are Shell Companies in Financial Services with low trading volumes. ##
## High transaction costs make these stocks expensive to trade in the real world. ##

,symbol,sector,industry,price,avg_volume,avg_sp,market_cap,rel_sp
1774,APXTU,Financial Services,Shell Companies,13.360000,8138.0,1.734300,5.351656e+08,0.129813
1970,HPK,Energy,Oil & Gas E&P,10.830000,3416.0,1.125539,9.926221e+08,0.103928
2468,THCAU,Financial Services,Shell Companies,11.010000,110.0,2.015000,2.420824e+08,0.183015
2473,FISK,Real Estate,REITÑDiversified,11.180000,589.0,1.176243,3.707329e+08,0.105210
2489,GNRSU,Financial Services,Shell Companies,10.760000,15775.0,1.372450,2.355633e+08,0.127551
2567,EXPCU,Financial Services,Shell Companies,12.850000,3874.0,1.726799,4.331250e+08,0.134381
2730,CRSAU,Financial Services,Shell Companies,10.640100,20.0,1.361818,3.153125e+08,0.127989
2800,CPAC,Basic Materials,Building Materials,8.010100,627.0,0.908621,7.319356e+08,0.113434
2827,NMMCU,Financial Services,Shell Companies,10.490000,4031.0,1.148472,1.656431e+08,0.109483
2887,VTRU,Consumer Defensive,Education & Training Services,14.200000,9729.0,1.472963,3.274243e+08,0.103730


In [9]:
# 1E. Dropping outliers with rel_sp > 0.1 from sp_stks #
sp_stks = sp_stks[sp_stks['rel_sp'] <= 0.1]
sp_stks.shape

(5521, 8)

In [10]:
# Question 2 #
# 2A. Importing statsmodels.formula.api as smf for regression analysis #
import statsmodels.formula.api as smf

In [11]:
# 2A_[i]. Running regression of avg_sp against price
lm2a_1 = smf.ols('avg_sp ~ price', data=sp_stks).fit()
lm2a_1.summary()

## Question: What can you conclude? ##
    ## Stock price is a strong predictor of the absolute bid-ask spread.
## For every $1 increase in price, the average spread increases by $0.0042.
## The relationship is highly significant (p < 0.001) with R-squared of 0.573. This means price explains 57.3% of the variation in spread. ##

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.573
Model:                            OLS   Adj. R-squared:                  0.573
Method:                 Least Squares   F-statistic:                     7410.
Date:                Sun, 23 Nov 2025   Prob (F-statistic):               0.00
Time:                        09:37:50   Log-Likelihood:                -3794.9
No. Observations:                5521   AIC:                             7594.
Df Residuals:                    5519   BIC:                             7607.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0224      0.007      3.260      0.001       0.009       0.036
price          0.0042    4.9e-05     86.084      0.000       0.004       0.004
==============================================================================
Omnibus:                     7741.247   Durbin-Watson:                   1.675
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         16150929.724
Skew:                           7.376   Prob(JB):                         0.00
Kurtosis:                     267.558   Cond. No.                         149.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [12]:
# 2a(ii). Running regression of rel_sp against price
lm2a_2 = smf.ols('rel_sp ~ price', data=sp_stks).fit()
lm2a_2.summary()

## Question: What can you conclude? ## 
    ## Stock price strongly predicts absolute spread but not relative spread.
## The negative coefficient shows that higher-priced stocks have slightly lower relative spreads. 
## In regression 1, price explains 57.3% of the variation in avg_sp.
## In regression 2, price explains only 1.2% of the variation in rel_sp.
## By dividing spread by price, rel_sp removes almost 98% of the mechanical price-spread relation price effect.
## This proves rel_sp is a better measure of liquidity independent of stock price

## Question: Compare the two regressions. ##
    ## Regression 1 (avg_sp ~ price):
## R-squared = 0.573
## Coefficient = 0.0042 (highly significant, p < 0.001)
## Price explains 57.3% of the variation in absolute spread
## Each $1 increase in price increases spread by $0.0042
    ## Regression 2 (rel_sp ~ price):
## R-squared = 0.012
## Coefficient = -8.201e-06 (still significant, p < 0.001)
## Price explains only 1.2% of the variation in relative spread
## Each $1 increase in price decreases rel_sp by 0.000008
## Weak negative relationship between price and relative spread

## Question: To what extent does the definition of rel_sp neutralize the impact of price on spread? ##
    ## rel_sp removes almost 98% of the price effect on spread.
## R-squared dropped from 57.3% to 1.2%.
## This makes rel_sp a far better measure for comparing liquidity across different price levels.

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 rel_sp   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     66.65
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           3.99e-16
Time:                        09:37:51   Log-Likelihood:                 17670.
No. Observations:                5521   AIC:                        -3.534e+04
Df Residuals:                    5519   BIC:                        -3.532e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0081      0.000     57.278      0.000       0.008       0.008
price      -8.201e-06      1e-06     -8.164      0.000   -1.02e-05   -6.23e-06
==============================================================================
Omnibus:                     4214.465   Durbin-Watson:                   1.745
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            87261.980
Skew:                           3.541   Prob(JB):                         0.00
Kurtosis:                      21.143   Cond. No.                         149.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [13]:
# 2B: Creating categories in price level #
import numpy as np

conditions = [
    (sp_stks['price'] <= 5),
    (sp_stks['price'] > 5) & (sp_stks['price'] <= 20),
    (sp_stks['price'] > 20) & (sp_stks['price'] <= 60),
    (sp_stks['price'] > 60) & (sp_stks['price'] <= 150), 
    (sp_stks['price'] > 150) 
]

levels = ['LL', 'L', 'M', 'H', 'HH']
sp_stks['p_lvl'] = np.select(conditions, levels)

## Verifying if the code works ##
sp_stks['p_lvl'].value_counts().sort_index()
    ## The code works ##

p_lvl
H      727
HH     332
L     2100
LL     832
M     1530
Name: count, dtype: int64

In [14]:
# 2C. Running a regression of avg_sp on the p_lvl #
lm2b = smf.ols('avg_sp ~ C(p_lvl)', data=sp_stks).fit()
lm2b.summary()

## Question: Interpretation of regression coefficients? ##
    ## Reference category is H (price $60-150) with an intercept = 0.3388
## HH (>$150): +0.84 higher spread than H
## L ($5-20): -0.23 lower spread than H  
## LL (≤$5): -0.31 lower spread than H
## M ($20-60): -0.11 lower spread than H
    ## Interpretation: Higher price levels have higher absolute spreads

## Question: Comparison of results with the 2A_[i] regression:
    ## Regression 2A_[i] (continuous price): R-squared = 0.573
## Each $1 increase in price increases spread by $0.0042
    ## Regression 2C (categorical p_lvl): R-squared = 0.124
## Shows the average spread differences between price groups
    ## The continuous price variable explains 46% more variation (57.3% vs 12.4%)
## Grouping into categories causes loss of the precise linear relationship
## Continuous price(2A_[i]) is superior for modeling the price-spread relationship

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.124
Model:                            OLS   Adj. R-squared:                  0.123
Method:                 Least Squares   F-statistic:                     195.3
Date:                Sun, 23 Nov 2025   Prob (F-statistic):          7.10e-157
Time:                        09:37:51   Log-Likelihood:                -5779.3
No. Observations:                5521   AIC:                         1.157e+04
Df Residuals:                    5516   BIC:                         1.160e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          0.3388      0.026     13.247      0.000       0.289       0.389
C(p_lvl)[T.HH]     0.8389      0.046     18.366      0.000       0.749       0.928
C(p_lvl)[T.L]     -0.2318      0.030     -7.812      0.000      -0.290      -0.174
C(p_lvl)[T.LL]    -0.3086      0.035     -8.815      0.000      -0.377      -0.240
C(p_lvl)[T.M]     -0.1109      0.031     -3.572      0.000      -0.172      -0.050
==============================================================================
Omnibus:                    12937.326   Durbin-Watson:                   1.921
Prob(Omnibus):                  0.000   Jarque-Bera (JB):        136452403.923
Skew:                          23.083   Prob(JB):                         0.00
Kurtosis:                     771.786   Cond. No.                         7.54
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [15]:
# Question 3 #
# 3A. Verifying code that filters for p_lvl == 'LL' #
P_LL=sp_stks[sp_stks['p_lvl']=='LL']
P_LL.shape

(832, 9)

In [16]:
# 3B #
Q1= P_LL['avg_volume'].quantile(0.25)
Q3= P_LL['avg_volume'].quantile(0.75)
cond_LL = [
    (P_LL['avg_volume'] <=  Q1),
    (P_LL['avg_volume'] >= Q1)  & (P_LL['avg_volume'] <= Q3),
    (P_LL['avg_volume'] >  Q3)  
    ]
adv_lvls_LL = ['L', 'M' , 'H' ]
P_LL['adv_lvl']=np.select(cond_LL,adv_lvls_LL)
P_LL['adv_lvl'].value_counts()

## Question: What does the code do? ##
    ## The code categorizes stocks by volume level:
## L: volume ≤ Q1 (low, 208 stocks)
## M: Q1 < volume ≤ Q3 (medium, 416 stocks)
## H: volume > Q3 (high, 208 stocks)

## Note: The code produces a SettingWithCopyWarning.## 
## This happens because P_LL is a view of sp_stks, not an independent copy.
## To fix this, we can Use P_LL = sp_stks[sp_stks['p_lvl'] == 'LL'].copy(). However, the code still works correctly and produces the expected output

/var/folders/17/ztgg_hjd4l1dh34jjv577pl00000gn/T/ipykernel_2800/3256700667.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_LL['adv_lvl']=np.select(cond_LL,adv_lvls_LL)


adv_lvl
M    416
H    208
L    208
Name: count, dtype: int64

In [17]:
# 3C_[i] #
lm3_a1 = smf.ols('avg_sp ~ adv_lvl -1'  , data = P_LL ).fit()
lm3_a1.summary()
## Question: Explain the output of the regression equation ##
    ## H: $0.0117, L: $0.0619, M: $0.0236 (all p < 0.001)
## Low volume stocks have 5.3×(0.0619 / 0.0117) wider spreads than high volume
## R-squared = 0.370
## Volume explains 37% of the spread variation in low-priced stocks

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.370
Model:                            OLS   Adj. R-squared:                  0.368
Method:                 Least Squares   F-statistic:                     243.3
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           7.41e-84
Time:                        09:37:51   Log-Likelihood:                 1897.5
No. Observations:                 832   AIC:                            -3789.
Df Residuals:                     829   BIC:                            -3775.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
adv_lvl[H]     0.0117      0.002      6.816      0.000       0.008       0.015
adv_lvl[L]     0.0619      0.002     36.044      0.000       0.059       0.065
adv_lvl[M]     0.0236      0.001     19.405      0.000       0.021       0.026
==============================================================================
Omnibus:                      711.436   Durbin-Watson:                   1.774
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            31593.974
Skew:                           3.593   Prob(JB):                         0.00
Kurtosis:                      32.321   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [18]:
# 3C_[ii] #
## Question: How -1 in the regression equation change the regression summary output? ##
    ## The -1 removes the intercept
## Without -1: one category is reference, others show differences
## With -1: all three categories appear with actual mean spreads (H, L, M)

## Question: Comment on explanation power of this equation ##
    ## R-squared = 0.370 (37% of variation explained)
## Moderate explanatory power
## Volume level is important but other factors also affect spreads

## Question: How avg_sp is related to adv_lvl. ##
    ## Inverse relationship: lower volume → higher spread
## L (low): $0.0619, M (medium): $0.0236, H (high): $0.0117
## Low volume stocks have 5.3× wider spreads than high volume


## Question: Can you interpret the regression output? ##
    ## All coefficients are highly significant (p < 0.001)
## Each coefficient represents actual mean spread for that volume level
## Trading volume strongly impacts transaction costs for low-priced stocks

In [19]:
# 3C_[iii]: Running regression with rel_sp instead of avg_sp #
lm3_b1 = smf.ols('rel_sp ~ adv_lvl -1', data=P_LL).fit()
lm3_b1.summary()
## Question: Is rel_sp related to adv_lvl? ##
    ## Yes, rel_sp is significantly related to adv_lvl
## All coefficients are highly significant (p < 0.001)
## H (high volume): 0.0057 (0.57%)
## L (low volume): 0.0185 (1.85%)
## M (medium volume): 0.0091 (0.91%)
## Low volume stocks have 3.2× higher relative spreads than high volume
## R-squared = 0.312 (31.2% explained)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 rel_sp   R-squared:                       0.312
Model:                            OLS   Adj. R-squared:                  0.310
Method:                 Least Squares   F-statistic:                     187.8
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           5.37e-68
Time:                        09:37:51   Log-Likelihood:                 2939.2
No. Observations:                 832   AIC:                            -5872.
Df Residuals:                     829   BIC:                            -5858.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
adv_lvl[H]     0.0057      0.000     11.622      0.000       0.005       0.007
adv_lvl[L]     0.0185      0.000     37.622      0.000       0.018       0.019
adv_lvl[M]     0.0091      0.000     26.148      0.000       0.008       0.010
==============================================================================
Omnibus:                      508.744   Durbin-Watson:                   1.818
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             8987.427
Skew:                           2.433   Prob(JB):                         0.00
Kurtosis:                      18.349   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [20]:
# 3D: Repeating the regression analysis for the other p_lvl values of L, M, H, HH #

# L Price Level #
P_L = sp_stks[sp_stks['p_lvl'] == 'L'].copy()
Q1 = P_L['avg_volume'].quantile(0.25)
Q3 = P_L['avg_volume'].quantile(0.75)
cond_L = [
    (P_L['avg_volume'] <= Q1),
    (P_L['avg_volume'] > Q1) & (P_L['avg_volume'] <= Q3),
    (P_L['avg_volume'] > Q3)
]
P_L['adv_lvl'] = np.select(cond_L, ['L', 'M', 'H'])

lm_L_avg = smf.ols('avg_sp ~ adv_lvl -1', data=P_L).fit()
lm_L_avg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.259
Model:                            OLS   Adj. R-squared:                  0.258
Method:                 Least Squares   F-statistic:                     366.8
Date:                Sun, 23 Nov 2025   Prob (F-statistic):          2.54e-137
Time:                        09:37:51   Log-Likelihood:                 1324.8
No. Observations:                2100   AIC:                            -2644.
Df Residuals:                    2097   BIC:                            -2627.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
adv_lvl[H]     0.0298      0.006      5.294      0.000       0.019       0.041
adv_lvl[L]     0.2335      0.006     41.529      0.000       0.223       0.245
adv_lvl[M]     0.0823      0.004     20.690      0.000       0.074       0.090
==============================================================================
Omnibus:                     1648.289   Durbin-Watson:                   1.908
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            48452.312
Skew:                           3.466   Prob(JB):                         0.00
Kurtosis:                      25.487   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [21]:
# Running regression on L price level by replacing avg_sp with rel_sp #
lm_L_rel = smf.ols('rel_sp ~ adv_lvl -1', data=P_L).fit()
lm_L_rel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 rel_sp   R-squared:                       0.281
Model:                            OLS   Adj. R-squared:                  0.281
Method:                 Least Squares   F-statistic:                     410.7
Date:                Sun, 23 Nov 2025   Prob (F-statistic):          3.19e-151
Time:                        09:37:51   Log-Likelihood:                 6689.2
No. Observations:                2100   AIC:                        -1.337e+04
Df Residuals:                    2097   BIC:                        -1.336e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
adv_lvl[H]     0.0026      0.000      6.024      0.000       0.002       0.003
adv_lvl[L]     0.0196      0.000     44.898      0.000       0.019       0.020
adv_lvl[M]     0.0076      0.000     24.544      0.000       0.007       0.008
==============================================================================
Omnibus:                     1245.286   Durbin-Watson:                   1.974
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            16416.338
Skew:                           2.557   Prob(JB):                         0.00
Kurtosis:                      15.707   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [22]:
# M Price Level #
P_M = sp_stks[sp_stks['p_lvl'] == 'M'].copy()
Q1 = P_M['avg_volume'].quantile(0.25)
Q3 = P_M['avg_volume'].quantile(0.75)
cond_M = [
    (P_M['avg_volume'] <= Q1),
    (P_M['avg_volume'] > Q1) & (P_M['avg_volume'] <= Q3),
    (P_M['avg_volume'] > Q3)
]
P_M['adv_lvl'] = np.select(cond_M, ['L', 'M', 'H'])

lm_M_avg = smf.ols('avg_sp ~ adv_lvl -1', data=P_M).fit()
lm_M_avg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.283
Model:                            OLS   Adj. R-squared:                  0.283
Method:                 Least Squares   F-statistic:                     302.1
Date:                Sun, 23 Nov 2025   Prob (F-statistic):          2.90e-111
Time:                        09:37:51   Log-Likelihood:                -175.89
No. Observations:                1530   AIC:                             357.8
Df Residuals:                    1527   BIC:                             373.8
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
adv_lvl[H]     0.0605      0.014      4.357      0.000       0.033       0.088
adv_lvl[L]     0.5133      0.014     36.968      0.000       0.486       0.540
adv_lvl[M]     0.1686      0.010     17.154      0.000       0.149       0.188
==============================================================================
Omnibus:                     1587.964   Durbin-Watson:                   1.849
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           121398.728
Skew:                           4.933   Prob(JB):                         0.00
Kurtosis:                      45.508   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [23]:
# Running regression on M price level by replacing avg_sp with rel_sp #
lm_M_rel = smf.ols('rel_sp ~ adv_lvl -1', data=P_M).fit()
lm_M_rel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 rel_sp   R-squared:                       0.340
Model:                            OLS   Adj. R-squared:                  0.339
Method:                 Least Squares   F-statistic:                     393.8
Date:                Sun, 23 Nov 2025   Prob (F-statistic):          1.18e-138
Time:                        09:37:51   Log-Likelihood:                 5285.4
No. Observations:                1530   AIC:                        -1.056e+04
Df Residuals:                    1527   BIC:                        -1.055e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
adv_lvl[H]     0.0016      0.000      4.213      0.000       0.001       0.002
adv_lvl[L]     0.0160      0.000     40.926      0.000       0.015       0.017
adv_lvl[M]     0.0047      0.000     16.804      0.000       0.004       0.005
==============================================================================
Omnibus:                     1402.150   Durbin-Watson:                   1.937
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            63741.769
Skew:                           4.189   Prob(JB):                         0.00
Kurtosis:                      33.491   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [24]:
# H Price Level #
P_H = sp_stks[sp_stks['p_lvl'] == 'H'].copy()
Q1 = P_H['avg_volume'].quantile(0.25)
Q3 = P_H['avg_volume'].quantile(0.75)
cond_H = [
    (P_H['avg_volume'] <= Q1),
    (P_H['avg_volume'] > Q1) & (P_H['avg_volume'] <= Q3),
    (P_H['avg_volume'] > Q3)
]
P_H['adv_lvl'] = np.select(cond_H, ['L', 'M', 'H'])

lm_H_avg = smf.ols('avg_sp ~ adv_lvl -1', data=P_H).fit()
lm_H_avg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.240
Model:                            OLS   Adj. R-squared:                  0.238
Method:                 Least Squares   F-statistic:                     114.3
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           6.98e-44
Time:                        09:37:51   Log-Likelihood:                -373.74
No. Observations:                 727   AIC:                             753.5
Df Residuals:                     724   BIC:                             767.2
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
adv_lvl[H]     0.1182      0.030      3.935      0.000       0.059       0.177
adv_lvl[L]     0.7196      0.030     23.945      0.000       0.661       0.779
adv_lvl[M]     0.2584      0.021     12.143      0.000       0.217       0.300
==============================================================================
Omnibus:                      996.257   Durbin-Watson:                   1.800
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           226380.020
Skew:                           7.193   Prob(JB):                         0.00
Kurtosis:                      88.243   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [25]:
# Running regression on H price level by replacing avg_sp with rel_sp #
lm_H_rel = smf.ols('rel_sp ~ adv_lvl -1', data=P_H).fit()
lm_H_rel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 rel_sp   R-squared:                       0.263
Model:                            OLS   Adj. R-squared:                  0.261
Method:                 Least Squares   F-statistic:                     129.1
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           1.13e-48
Time:                        09:37:51   Log-Likelihood:                 2933.9
No. Observations:                 727   AIC:                            -5862.
Df Residuals:                     724   BIC:                            -5848.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
adv_lvl[H]     0.0012      0.000      3.848      0.000       0.001       0.002
adv_lvl[L]     0.0080      0.000     25.054      0.000       0.007       0.009
adv_lvl[M]     0.0028      0.000     12.249      0.000       0.002       0.003
==============================================================================
Omnibus:                     1015.926   Durbin-Watson:                   1.760
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           260451.712
Skew:                           7.419   Prob(JB):                         0.00
Kurtosis:                      94.531   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [26]:
# HH Price Level #
P_HH = sp_stks[sp_stks['p_lvl'] == 'HH'].copy()
Q1 = P_HH['avg_volume'].quantile(0.25)
Q3 = P_HH['avg_volume'].quantile(0.75)
cond_HH = [
    (P_HH['avg_volume'] <= Q1),
    (P_HH['avg_volume'] > Q1) & (P_HH['avg_volume'] <= Q3),
    (P_HH['avg_volume'] > Q3)
]
P_HH['adv_lvl'] = np.select(cond_HH, ['L', 'M', 'H'])

lm_HH_avg = smf.ols('avg_sp ~ adv_lvl -1', data=P_HH).fit()
lm_HH_avg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.134
Model:                            OLS   Adj. R-squared:                  0.129
Method:                 Least Squares   F-statistic:                     25.51
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           5.03e-11
Time:                        09:37:51   Log-Likelihood:                -765.62
No. Observations:                 332   AIC:                             1537.
Df Residuals:                     329   BIC:                             1549.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
adv_lvl[H]     0.4561      0.268      1.703      0.089      -0.071       0.983
adv_lvl[L]     2.8236      0.268     10.546      0.000       2.297       3.350
adv_lvl[M]     0.7155      0.189      3.779      0.000       0.343       1.088
==============================================================================
Omnibus:                      458.436   Durbin-Watson:                   2.086
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            51820.155
Skew:                           6.748   Prob(JB):                         0.00
Kurtosis:                      62.698   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [27]:
# Running regression on HH price level by replacing avg_sp with rel_sp #
lm_HH_rel = smf.ols('rel_sp ~ adv_lvl -1', data=P_HH).fit()
lm_HH_rel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 rel_sp   R-squared:                       0.394
Model:                            OLS   Adj. R-squared:                  0.390
Method:                 Least Squares   F-statistic:                     106.9
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           1.68e-36
Time:                        09:37:51   Log-Likelihood:                 1533.3
No. Observations:                 332   AIC:                            -3061.
Df Residuals:                     329   BIC:                            -3049.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
adv_lvl[H]     0.0013      0.000      4.885      0.000       0.001       0.002
adv_lvl[L]     0.0063      0.000     23.923      0.000       0.006       0.007
adv_lvl[M]     0.0023      0.000     12.294      0.000       0.002       0.003
==============================================================================
Omnibus:                      258.711   Durbin-Watson:                   1.811
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             5150.956
Skew:                           3.044   Prob(JB):                         0.00
Kurtosis:                      21.311   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [28]:
## Question: What can you conclude about the importance of average volume on a stock’s spread and relative spread?
    ## Average volume is critical for both absolute and relative spreads across all price levels
## All coefficients highly significant (p < 0.001) - volume strongly predicts spreads
## Consistent inverse relationship: Low volume → wider spreads, High volume → narrower spreads
## Effect varies by price: Lower-priced stocks (L: R²=0.259-0.281) show stronger volume effects than higher-priced stocks (HH: R²=0.134-0.394)
## Relative spread analysis confirms pattern holds even when controlling for price differences
## Bottom line: Trading volume fundamentally determines transaction costs - low liquidity means higher costs regardless of stock price

In [29]:
# Question 4 #

# 4A: Testing if market_cap adds explanatory power to explaining a stock’s spread #
# Model without market_cap 
lm3_a1 = smf.ols('avg_sp ~ adv_lvl -1', data=P_LL).fit()
lm3_a1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.370
Model:                            OLS   Adj. R-squared:                  0.368
Method:                 Least Squares   F-statistic:                     243.3
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           7.41e-84
Time:                        09:37:51   Log-Likelihood:                 1897.5
No. Observations:                 832   AIC:                            -3789.
Df Residuals:                     829   BIC:                            -3775.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
adv_lvl[H]     0.0117      0.002      6.816      0.000       0.008       0.015
adv_lvl[L]     0.0619      0.002     36.044      0.000       0.059       0.065
adv_lvl[M]     0.0236      0.001     19.405      0.000       0.021       0.026
==============================================================================
Omnibus:                      711.436   Durbin-Watson:                   1.774
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            31593.974
Skew:                           3.593   Prob(JB):                         0.00
Kurtosis:                      32.321   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [30]:
# Model with market_cap
lm4_a = smf.ols('avg_sp ~ adv_lvl + market_cap -1', data=P_LL).fit()
lm4_a.summary()
 
# Question: Does market_cap add explanation power for explaining a stock’s spread?
    ## Market cap does not add explanatory power for predicting price spreads
## R² increased minimally: 0.370 → 0.371 (0.1% improvement)
## market_cap coefficient: -2.43e-13, p = 0.237 (NOT significant, p > 0.05)
## Volume level by itself is sufficient, adding market_cap provides no additional predictive value to our model

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.371
Model:                            OLS   Adj. R-squared:                  0.369
Method:                 Least Squares   F-statistic:                     162.7
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           6.52e-83
Time:                        09:37:51   Log-Likelihood:                 1898.2
No. Observations:                 832   AIC:                            -3788.
Df Residuals:                     828   BIC:                            -3770.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
adv_lvl[H]     0.0121      0.002      6.914      0.000       0.009       0.016
adv_lvl[L]     0.0620      0.002     36.072      0.000       0.059       0.065
adv_lvl[M]     0.0237      0.001     19.446      0.000       0.021       0.026
market_cap  -2.43e-13   2.06e-13     -1.182      0.237   -6.47e-13     1.6e-13
==============================================================================
Omnibus:                      711.766   Durbin-Watson:                   1.769
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            31661.336
Skew:                           3.595   Prob(JB):                         0.00
Kurtosis:                      32.353   Cond. No.                     8.76e+09
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 8.76e+09. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [31]:
# 4B #
## Application to Portfolio Trading Cost Estimation:
    #1. Use price and volume to classify each stock (p_lvl and adv_lvl)
    #2. Apply regression coefficients to predict the expected spread for each stock
    #3. Average spread represents the minimum transaction cost per share

## Stock-Specific Lower Bound on Trading Cost:
## For a stock with price P and volume V:
## - Classify: p_lvl (price category) and adv_lvl (volume category)
## - Lower bound = predicted avg_sp from regression
## - Example: LL price, Low volume → min cost = $0.2335 per share
## - High volume stocks always have lower costs (High vol ≈ $0.01-0.03 across price levels)

## This provides a baseline cost estimate before considering market impact and timing

# 4B: How this analysis helps portfolio managers estimate trading costs #
    ## A portfolio manager can use the regression coefficients to estimate trading costs:
    ## Identify the stock's price level (LL, L, M, H, HH) and volume level (L, M, H)
    ## Use the corresponding coefficient from the regression as the expected avg_sp
    ## For a stock-specific lower bound: use the stock's actual avg_sp value
    ## Example: For a low-price, high-volume stock (LL-H), expect ~$0.0117 spread
    ## Multiply by shares traded to estimate total transaction cost from spreads
    ## This provides a baseline estimate before executing trades

In [32]:
# QUESTION 5: Repeating analysis for ETFs #
    # Reading in ETF and preparing data
etfs = pd.read_csv('ets_spread.csv')
etfs['p_lvl'] = pd.cut(etfs['price'], bins=[0, 20, 60, 150, np.inf], labels=['L', 'M', 'H', 'HH'])
etfs['rel_sp'] = etfs['avg_sp'] / etfs['price']

In [33]:
# Checking the structure of the data
etfs.shape

(1911, 8)

In [62]:
# Creating price categories for all ETFs: LL, L, M, H, HH 
conditions = [
    etfs['price'] <= 5,
    (etfs['price'] > 5) & (etfs['price'] <= 20),
    (etfs['price'] > 20) & (etfs['price'] <= 60),
    (etfs['price'] > 60) & (etfs['price'] <= 150),
    etfs['price'] > 150
]
levels = ['LL', 'L', 'M', 'H', 'HH']
etfs['p_lvl'] = np.select(conditions, levels)
etfs['p_lvl'].value_counts().sort_index()

p_lvl
H      430
HH      99
L      201
LL      12
M     1169
Name: count, dtype: int64

In [68]:
# Create volume categories for all ETFs
Q1 = etfs['avg_volume'].quantile(0.25)
Q3 = etfs['avg_volume'].quantile(0.75)
etfs['adv_lvl'] = np.select([
    (etfs['avg_volume'] <= Q1),
    (etfs['avg_volume'] > Q1) & (etfs['avg_volume'] <= Q3),
    (etfs['avg_volume'] > Q3)
], ['L', 'M', 'H'])

In [70]:
# Regression for ETFs - avg_sp
lm_etf = smf.ols('avg_sp ~ adv_lvl -1', data=etfs).fit()
lm_etf.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.043
Model:                            OLS   Adj. R-squared:                  0.042
Method:                 Least Squares   F-statistic:                     42.77
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           6.78e-19
Time:                        10:20:32   Log-Likelihood:                -644.20
No. Observations:                1911   AIC:                             1294.
Df Residuals:                    1908   BIC:                             1311.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
adv_lvl[H]     0.0330      0.016      2.129      0.033       0.003       0.063
adv_lvl[L]     0.2344      0.016     15.107      0.000       0.204       0.265
adv_lvl[M]     0.1159      0.011     10.561      0.000       0.094       0.137
==============================================================================
Omnibus:                     4573.819   Durbin-Watson:                   2.015
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         44909535.859
Skew:                          23.532   Prob(JB):                         0.00
Kurtosis:                     752.532   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [72]:
# Regression for ETFs - rel_sp
lm_etf_rel = smf.ols('rel_sp ~ C(adv_lvl) - 1', data=etfs).fit()
lm_etf_rel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 rel_sp   R-squared:                       0.106
Model:                            OLS   Adj. R-squared:                  0.105
Method:                 Least Squares   F-statistic:                     112.6
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           5.90e-47
Time:                        10:20:35   Log-Likelihood:                 7425.1
No. Observations:                1911   AIC:                        -1.484e+04
Df Residuals:                    1908   BIC:                        -1.483e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
C(adv_lvl)[H]     0.0007      0.000      3.261      0.001       0.000       0.001
C(adv_lvl)[L]     0.0055      0.000     24.208      0.000       0.005       0.006
C(adv_lvl)[M]     0.0026      0.000     16.003      0.000       0.002       0.003
==============================================================================
Omnibus:                     3454.052   Durbin-Watson:                   1.972
Prob(Omnibus):                  0.000   Jarque-Bera (JB):          5241233.239
Skew:                          12.668   Prob(JB):                         0.00
Kurtosis:                     258.308   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [108]:
# P_LL: Very low price ETFs
P_LL_etf = etfs[etfs['p_lvl'] == 'LL'].copy()
Q1 = P_LL_etf['avg_volume'].quantile(0.25)
Q3 = P_LL_etf['avg_volume'].quantile(0.75)
P_LL_etf['adv_lvl'] = np.select([
    P_LL_etf['avg_volume'] <= Q1,
    (P_LL_etf['avg_volume'] > Q1) & (P_LL_etf['avg_volume'] <= Q3),
    P_LL_etf['avg_volume'] > Q3
], ['L', 'M', 'H'])

# Regression for ETF P_LL: avg_sp #
lm_LL_etf_avg = smf.ols('avg_sp ~ C(adv_lvl) - 1', data=P_LL_etf).fit()
lm_LL_etf_avg.summary()

/opt/anaconda3/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=12
  res = hypotest_fun_out(*samples, **kwds)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.287
Model:                            OLS   Adj. R-squared:                  0.129
Method:                 Least Squares   F-statistic:                     1.813
Date:                Sun, 23 Nov 2025   Prob (F-statistic):              0.218
Time:                        10:29:06   Log-Likelihood:                 26.250
No. Observations:                  12   AIC:                            -46.50
Df Residuals:                       9   BIC:                            -45.05
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
C(adv_lvl)[H]     0.0123      0.018      0.679      0.514      -0.029       0.053
C(adv_lvl)[L]     0.0592      0.018      3.274      0.010       0.018       0.100
C(adv_lvl)[M]     0.0265      0.013      2.074      0.068      -0.002       0.055
==============================================================================
Omnibus:                        4.007   Durbin-Watson:                   2.788
Prob(Omnibus):                  0.135   Jarque-Bera (JB):                1.640
Skew:                           0.879   Prob(JB):                        0.440
Kurtosis:                       3.436   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [110]:
# Regression for ETF P_LL: rel_sp #
lm_LL_etf_rel = smf.ols('rel_sp ~ C(adv_lvl) - 1', data=P_LL_etf).fit()
lm_LL_etf_rel.summary()

/opt/anaconda3/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=12
  res = hypotest_fun_out(*samples, **kwds)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 rel_sp   R-squared:                       0.405
Model:                            OLS   Adj. R-squared:                  0.273
Method:                 Least Squares   F-statistic:                     3.067
Date:                Sun, 23 Nov 2025   Prob (F-statistic):             0.0964
Time:                        10:29:07   Log-Likelihood:                 42.732
No. Observations:                  12   AIC:                            -79.46
Df Residuals:                       9   BIC:                            -78.01
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
C(adv_lvl)[H]     0.0038      0.005      0.818      0.434      -0.007       0.014
C(adv_lvl)[L]     0.0193      0.005      4.215      0.002       0.009       0.030
C(adv_lvl)[M]     0.0088      0.003      2.702      0.024       0.001       0.016
==============================================================================
Omnibus:                        1.663   Durbin-Watson:                   2.931
Prob(Omnibus):                  0.435   Jarque-Bera (JB):                0.243
Skew:                          -0.286   Prob(JB):                        0.886
Kurtosis:                       3.397   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [112]:
# P_L: Low price ETFs #
P_L_etf = etfs[etfs['p_lvl'] == 'L'].copy()
Q1 = P_L_etf['avg_volume'].quantile(0.25)
Q3 = P_L_etf['avg_volume'].quantile(0.75)
P_L_etf['adv_lvl'] = np.select([
    P_L_etf['avg_volume'] <= Q1,
    (P_L_etf['avg_volume'] > Q1) & (P_L_etf['avg_volume'] <= Q3),
    P_L_etf['avg_volume'] > Q3
], ['L', 'M', 'H'])

# Regression for ETF P_L: avg_sp #
lm_L_etf_avg = smf.ols('avg_sp ~ C(adv_lvl) - 1', data=P_L_etf).fit()
lm_L_etf_avg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.232
Model:                            OLS   Adj. R-squared:                  0.224
Method:                 Least Squares   F-statistic:                     29.87
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           4.60e-12
Time:                        10:29:08   Log-Likelihood:                 291.51
No. Observations:                 201   AIC:                            -577.0
Df Residuals:                     198   BIC:                            -567.1
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
C(adv_lvl)[H]     0.0150      0.008      1.852      0.065      -0.001       0.031
C(adv_lvl)[L]     0.1028      0.008     12.846      0.000       0.087       0.119
C(adv_lvl)[M]     0.0567      0.006      9.915      0.000       0.045       0.068
==============================================================================
Omnibus:                      124.047   Durbin-Watson:                   1.889
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              742.219
Skew:                           2.424   Prob(JB):                    6.75e-162
Kurtosis:                      11.070   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [114]:
# Regression for ETF P_L: rel_sp #
lm_L_etf_rel = smf.ols('rel_sp ~ C(adv_lvl) - 1', data=P_L_etf).fit()
lm_L_etf_rel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 rel_sp   R-squared:                       0.179
Model:                            OLS   Adj. R-squared:                  0.170
Method:                 Least Squares   F-statistic:                     21.54
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           3.43e-09
Time:                        10:29:08   Log-Likelihood:                 787.17
No. Observations:                 201   AIC:                            -1568.
Df Residuals:                     198   BIC:                            -1558.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
C(adv_lvl)[H]     0.0012      0.001      1.727      0.086      -0.000       0.003
C(adv_lvl)[L]     0.0075      0.001     11.058      0.000       0.006       0.009
C(adv_lvl)[M]     0.0041      0.000      8.500      0.000       0.003       0.005
==============================================================================
Omnibus:                      207.557   Durbin-Watson:                   1.863
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             5599.278
Skew:                           4.047   Prob(JB):                         0.00
Kurtosis:                      27.557   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [116]:
# P_M (Medium price ETFs)
P_M_etf = etfs[etfs['p_lvl'] == 'M'].copy()
Q1 = P_M_etf['avg_volume'].quantile(0.25)
Q3 = P_M_etf['avg_volume'].quantile(0.75)
P_M_etf['adv_lvl'] = np.select([
    P_M_etf['avg_volume'] <= Q1,
    (P_M_etf['avg_volume'] > Q1) & (P_M_etf['avg_volume'] <= Q3),
    P_M_etf['avg_volume'] > Q3
], ['L', 'M', 'H'])

# Regression for ETF P_M: avg_sp #
lm_M_etf_avg = smf.ols('avg_sp ~ C(adv_lvl) - 1', data=P_M_etf).fit()
lm_M_etf_avg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.121
Model:                            OLS   Adj. R-squared:                  0.120
Method:                 Least Squares   F-statistic:                     80.63
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           1.60e-33
Time:                        10:29:08   Log-Likelihood:                 630.64
No. Observations:                1169   AIC:                            -1255.
Df Residuals:                    1166   BIC:                            -1240.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
C(adv_lvl)[H]     0.0306      0.008      3.698      0.000       0.014       0.047
C(adv_lvl)[L]     0.1788      0.008     21.669      0.000       0.163       0.195
C(adv_lvl)[M]     0.1015      0.006     17.362      0.000       0.090       0.113
==============================================================================
Omnibus:                     1842.542   Durbin-Watson:                   2.026
Prob(Omnibus):                  0.000   Jarque-Bera (JB):          1423470.382
Skew:                           9.348   Prob(JB):                         0.00
Kurtosis:                     172.926   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [118]:
# Regression for ETF P_M: rel_sp #
lm_M_etf_rel = smf.ols('rel_sp ~ C(adv_lvl) - 1', data=P_M_etf).fit()
lm_M_etf_rel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 rel_sp   R-squared:                       0.111
Model:                            OLS   Adj. R-squared:                  0.109
Method:                 Least Squares   F-statistic:                     72.57
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           1.97e-30
Time:                        10:29:08   Log-Likelihood:                 4649.6
No. Observations:                1169   AIC:                            -9293.
Df Residuals:                    1166   BIC:                            -9278.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
C(adv_lvl)[H]     0.0009      0.000      3.218      0.001       0.000       0.001
C(adv_lvl)[L]     0.0054      0.000     20.226      0.000       0.005       0.006
C(adv_lvl)[M]     0.0029      0.000     15.278      0.000       0.003       0.003
==============================================================================
Omnibus:                     2133.238   Durbin-Watson:                   2.044
Prob(Omnibus):                  0.000   Jarque-Bera (JB):          3513305.283
Skew:                          12.489   Prob(JB):                         0.00
Kurtosis:                     270.405   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [120]:
# P_H (High price ETFs)
P_H_etf = etfs[etfs['p_lvl'] == 'H'].copy()
Q1 = P_H_etf['avg_volume'].quantile(0.25)
Q3 = P_H_etf['avg_volume'].quantile(0.75)
P_H_etf['adv_lvl'] = np.select([
    P_H_etf['avg_volume'] <= Q1,
    (P_H_etf['avg_volume'] > Q1) & (P_H_etf['avg_volume'] <= Q3),
    P_H_etf['avg_volume'] > Q3
], ['L', 'M', 'H'])

# Regression for ETF P_H: avg_sp #
lm_H_etf_avg = smf.ols('avg_sp ~ C(adv_lvl) - 1', data=P_H_etf).fit()
lm_H_etf_avg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.053
Model:                            OLS   Adj. R-squared:                  0.049
Method:                 Least Squares   F-statistic:                     12.03
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           8.26e-06
Time:                        10:29:09   Log-Likelihood:                -391.54
No. Observations:                 430   AIC:                             789.1
Df Residuals:                     427   BIC:                             801.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
C(adv_lvl)[H]     0.0302      0.058      0.520      0.603      -0.084       0.144
C(adv_lvl)[L]     0.4126      0.058      7.104      0.000       0.298       0.527
C(adv_lvl)[M]     0.1314      0.041      3.185      0.002       0.050       0.213
==============================================================================
Omnibus:                      935.075   Durbin-Watson:                   1.898
Prob(Omnibus):                  0.000   Jarque-Bera (JB):          1677828.793
Skew:                          16.319   Prob(JB):                         0.00
Kurtosis:                     307.271   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [122]:
# Regression for ETF P_H: rel_sp #
lm_H_etf_rel = smf.ols('rel_sp ~ C(adv_lvl) - 1', data=P_H_etf).fit()
lm_H_etf_rel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 rel_sp   R-squared:                       0.062
Model:                            OLS   Adj. R-squared:                  0.058
Method:                 Least Squares   F-statistic:                     14.18
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           1.09e-06
Time:                        10:29:09   Log-Likelihood:                 1580.5
No. Observations:                 430   AIC:                            -3155.
Df Residuals:                     427   BIC:                            -3143.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
C(adv_lvl)[H]     0.0003      0.001      0.542      0.588      -0.001       0.001
C(adv_lvl)[L]     0.0046      0.001      7.733      0.000       0.003       0.006
C(adv_lvl)[M]     0.0015      0.000      3.595      0.000       0.001       0.002
==============================================================================
Omnibus:                      908.000   Durbin-Watson:                   1.881
Prob(Omnibus):                  0.000   Jarque-Bera (JB):          1383166.931
Skew:                          15.312   Prob(JB):                         0.00
Kurtosis:                     279.156   Cond. No.                         1.41
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [124]:
# P_HH (Very high price ETFs)
P_HH_etf = etfs[etfs['p_lvl'] == 'HH'].copy()
Q1 = P_HH_etf['avg_volume'].quantile(0.25)
Q3 = P_HH_etf['avg_volume'].quantile(0.75)
P_HH_etf['adv_lvl'] = np.select([
    P_HH_etf['avg_volume'] <= Q1,
    (P_HH_etf['avg_volume'] > Q1) & (P_HH_etf['avg_volume'] <= Q3),
    P_HH_etf['avg_volume'] > Q3
], ['L', 'M', 'H'])

# Regression for ETF P_HH: avg_sp #
lm_HH_etf_avg = smf.ols('avg_sp ~ C(adv_lvl) - 1', data=P_HH_etf).fit()
lm_HH_etf_avg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 avg_sp   R-squared:                       0.138
Model:                            OLS   Adj. R-squared:                  0.120
Method:                 Least Squares   F-statistic:                     7.678
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           0.000807
Time:                        10:29:09   Log-Likelihood:                -77.098
No. Observations:                  99   AIC:                             160.2
Df Residuals:                      96   BIC:                             168.0
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
C(adv_lvl)[H]     0.0604      0.107      0.564      0.574      -0.152       0.273
C(adv_lvl)[L]     0.6387      0.107      5.965      0.000       0.426       0.851
C(adv_lvl)[M]     0.2551      0.076      3.335      0.001       0.103       0.407
==============================================================================
Omnibus:                      155.069   Durbin-Watson:                   2.356
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             8026.902
Skew:                           5.725   Prob(JB):                         0.00
Kurtosis:                      45.601   Cond. No.                         1.40
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [126]:
# Regression for ETF P_HH: rel_sp #
lm_HH_etf_rel = smf.ols('rel_sp ~ C(adv_lvl) - 1', data=P_HH_etf).fit()
lm_HH_etf_rel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 rel_sp   R-squared:                       0.257
Model:                            OLS   Adj. R-squared:                  0.242
Method:                 Least Squares   F-statistic:                     16.61
Date:                Sun, 23 Nov 2025   Prob (F-statistic):           6.39e-07
Time:                        10:29:09   Log-Likelihood:                 514.14
No. Observations:                  99   AIC:                            -1022.
Df Residuals:                      96   BIC:                            -1014.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
C(adv_lvl)[H]     0.0002      0.000      0.914      0.363      -0.000       0.001
C(adv_lvl)[L]     0.0024      0.000      8.948      0.000       0.002       0.003
C(adv_lvl)[M]     0.0011      0.000      5.534      0.000       0.001       0.001
==============================================================================
Omnibus:                       69.111   Durbin-Watson:                   2.178
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              331.875
Skew:                           2.369   Prob(JB):                     8.59e-73
Kurtosis:                      10.616   Cond. No.                         1.40
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [128]:
## Answer: Question 5 ##
    ## Comparison: ETFs vs Stocks
## ETFs: R² = 0.084 (volume explains only 8.4% of spread variation)
## Stocks: R² = 0.370 (volume explains 37% of spread variation)
## Key difference: ETF creation/redemption mechanism provides liquidity, reducing volume's impact on spreads

In [131]:
# Question 6 #
